# 02b · América Latina: datos para la pregunta final

**Pregunta del equipo:** *Entre 2006 y 2025, mientras bajaban la fecundidad y la mortalidad infantil en América Latina, ¿se redujo o creció la desigualdad de aprendizaje dentro de los países?*

**Objetivo:** preparar las dos tablas que necesita el análisis, con las decisiones tomadas **antes** de mirar resultados (27 de octubre de 2026):

| Decisión | Elección | Por qué |
|---|---|---|
| Fuente de PISA | Tablas oficiales de tendencia de la OCDE (*PISA 2025 Results, Vol. I*, tablas I.B1.2a.33–41) | Una sola fuente, valores revisados por la OCDE, comparables entre ciclos y con error estándar |
| Desigualdad de aprendizaje | Brecha **P90 − P10** y **% bajo el nivel 2** | La primera mide la dispersión; la segunda, cuántos jóvenes no alcanzan el mínimo |
| Materia | Matemáticas como principal; lectura y ciencias para comprobar robustez | Matemáticas tiene la serie más larga y comparable |
| Países | Panel de 8: Argentina, Brasil, Chile, Colombia, Costa Rica, México, Perú y Uruguay | Son los que tienen PISA en los 6 ciclos de 2009 a 2025 |
| Período | 2009–2025 (2006 solo como contexto) | 2009 es el primer ciclo con los 8 países |
| Argentina 2015 | Se excluye ese ciclo | La OCDE lo considera no comparable: la reestructuración de la secundaria redujo la cobertura de la muestra |
| Cobertura escolar | Se agrega el índice de cobertura 3 de la OCDE (tabla I.A2.1) | Si más jóvenes de 15 años llegan escolarizados, la desigualdad medida puede cambiar por eso y no por el sistema educativo |

**Salidas** (en `data/processed/`):

| Archivo | Una fila es... | Para qué |
|---|---|---|
| `al_wpp_1950_2100.csv` | un país de América Latina (o la región) en un año, 1950–2100 | Las trayectorias de fecundidad y mortalidad infantil, con intervalos de predicción desde 2024 |
| `al_panel_pisa.csv` | uno de los 8 países, en un ciclo de PISA y una materia | La tabla principal del análisis: desigualdad de aprendizaje + cobertura + demografía |

In [1]:
%load_ext autoreload
%autoreload 2

import re
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # permite importar src/

import numpy as np
import pandas as pd

from src.utils import DATA_RAW, cargar_raw, guardar_processed

pd.set_option("display.max_columns", None)

ULTIMO_ESTIMADO = 2023  # desde 2024, WPP2024 son proyecciones

# Panel principal: los 8 países con PISA en los 6 ciclos de 2009 a 2025
PANEL = {
    "ARG": "Argentina", "BRA": "Brazil", "CHL": "Chile", "COL": "Colombia",
    "CRI": "Costa Rica", "MEX": "Mexico", "PER": "Peru", "URY": "Uruguay",
}
CICLOS = [2006, 2009, 2012, 2015, 2018, 2022, 2025]
INICIO_PANEL = 2009
EXCLUIR = {("ARG", 2015): "La OCDE considera no comparable el ciclo 2015 de Argentina (cobertura de la muestra)"}

## 1. Base de la ONU: América Latina de 1950 a 2100

América Latina y el Caribe son las subregiones *Caribbean* (915), *Central America* (916) y *South America* (931). Se guardan todos sus países (para dar contexto) y el agregado regional `Latin America and the Caribbean` (`LocID` 904), que es la suma de esos países según la ONU.

Indicadores: fecundidad (`TFR`), mortalidad infantil (`IMR`), maternidad adolescente (`pct_births_adolescentes = 100 × Births1519 / Births`), edad mediana y esperanza de vida, como contexto.

In [2]:
dem = cargar_raw(
    "WPP2024_Demographic_Indicators_Medium_csv.gz",
    compression="gzip", encoding="utf-8-sig", low_memory=False,
)
SUBREGIONES_AL = {915: "Caribbean", 916: "Central America", 931: "South America"}
REGION_AL = 904

es_pais_al = dem["ParentID"].isin(SUBREGIONES_AL) & dem["ISO3_code"].notna()
al = dem.loc[(es_pais_al | dem["LocID"].eq(REGION_AL)) & dem["Time"].le(2100)].copy()
al["Subregion"] = al["ParentID"].map(SUBREGIONES_AL)
al.loc[al["LocID"].eq(REGION_AL), ["ISO3_code", "Subregion"]] = ["AL_REGION", "Región"]

COLS = ["TFR", "IMR", "Births", "Births1519", "MedianAgePop", "LEx", "TPopulation1July", "CNMR"]
al = al[["LocID", "ISO3_code", "Location", "Subregion", "Time"] + COLS]
al["pct_births_adolescentes"] = 100 * al["Births1519"] / al["Births"]
al["es_proyeccion"] = al["Time"] > ULTIMO_ESTIMADO
al["en_panel_pisa"] = al["ISO3_code"].isin(PANEL)

print("Países de América Latina:", al.loc[al["ISO3_code"] != "AL_REGION", "ISO3_code"].nunique())
print("Años:", al["Time"].min(), "-", al["Time"].max())
assert set(PANEL) <= set(al["ISO3_code"]), "Falta algún país del panel"

# El agregado regional debe ser la suma de los países (comprobación de doble conteo)
suma = al.loc[al["ISO3_code"] != "AL_REGION"].groupby("Time")["TPopulation1July"].sum()
region = al.loc[al["ISO3_code"] == "AL_REGION"].set_index("Time")["TPopulation1July"]
print(f"Máx. |suma de países - región| (miles): {(suma - region).abs().max():.3f}")
assert (suma - region).abs().max() < 1

Países de América Latina: 50
Años: 1950 - 2100
Máx. |suma de países - región| (miles): 0.008


### Incertidumbre de las proyecciones

Para 2024–2100 se agregan los intervalos de predicción del 80 % y del 95 % de `OtherVariants` para `TFR` e `IMR`. Así, cualquier gráfico que muestre el futuro puede mostrar también cuánto se sabe de él.

In [3]:
var = cargar_raw(
    "WPP2024_Demographic_Indicators_OtherVariants_csv.gz",
    compression="gzip", encoding="utf-8-sig", low_memory=False,
    usecols=["LocID", "Variant", "Time", "TFR", "IMR"],
)
PI = {"Lower 95 PI": "PI95_inf", "Lower 80 PI": "PI80_inf", "Upper 80 PI": "PI80_sup", "Upper 95 PI": "PI95_sup"}
pi = (
    var.loc[var["Variant"].isin(PI) & var["LocID"].isin(al["LocID"]) & var["Time"].le(2100)]
    .pivot_table(index=["LocID", "Time"], columns="Variant", values=["TFR", "IMR"])
)
pi.columns = [f"{ind}_{PI[v]}" for ind, v in pi.columns]
pi = pi.reset_index()

antes = len(al)
al = al.merge(pi, on=["LocID", "Time"], how="left", validate="one_to_one")
assert len(al) == antes

# Coherencia: los intervalos deben envolver a la variante media
for ind in ["TFR", "IMR"]:
    p = al.dropna(subset=[f"{ind}_PI95_inf"])
    fuera = ((p[ind] < p[f"{ind}_PI95_inf"] - 1e-3) | (p[ind] > p[f"{ind}_PI95_sup"] + 1e-3)).sum()
    print(f"{ind}: filas con intervalo | años {int(p['Time'].min())}-{int(p['Time'].max())} | valor medio fuera del 95 %: {fuera}")

TFR: filas con intervalo | años 2024-2100 | valor medio fuera del 95 %: 0
IMR: filas con intervalo | años 2024-2100 | valor medio fuera del 95 %: 0


## 2. PISA: tablas oficiales de tendencia de la OCDE

Las tablas I.B1.2a.33 a 2a.41 del libro de PISA 2025 traen, para cada país y ciclo, los percentiles 10, 25, 50, 75 y 90 y el porcentaje de estudiantes bajo el nivel 2, cada uno con su error estándar. Tienen tres filas de encabezado (ciclo, medida y tipo de valor), así que se leen sin encabezado y se reconstruyen.

| Materia | Distribución (percentiles) | Bajo el nivel 2 |
|---|---|---|
| Matemáticas | I.B1.2a.41 (2003–2025) | I.B1.2a.35 (2003–2025) |
| Lectura | I.B1.2a.40 (2000–2025) | I.B1.2a.34 (2000–2025) |
| Ciencias | I.B1.2a.39 (2006–2025) | I.B1.2a.33 (2006–2025) |

En estas tablas, `m` significa dato no disponible y se convierte en vacío.

In [4]:
EXCEL_TENDENCIAS = "PISA2025_cap2a_desempeno_niveles.xlsx"


def limpiar_nombre(nombre):
    # Quita asteriscos y números de nota al pie: "Canada*" -> "Canada"
    return re.sub(r"[\*\d]+$", "", str(nombre)).strip()


def leer_tendencia(hoja):
    # Devuelve una tabla larga: país, ciclo, medida, valor y error estándar
    t = pd.read_excel(DATA_RAW / EXCEL_TENDENCIAS, sheet_name=hoja, header=None)
    # Fila de ciclos: la primera con una celda que termina en "PISA 20xx" (no el título)
    fila_ciclo = next(i for i in range(len(t)) if t.iloc[i].astype(str).str.strip().str.contains(r"PISA 20\d\d$").any())
    ciclo = t.iloc[fila_ciclo].ffill()
    medida = t.iloc[fila_ciclo + 1].ffill()
    tipo = t.iloc[fila_ciclo + 2]

    filas = []
    paises = t.iloc[fila_ciclo + 3:, 0].map(limpiar_nombre)
    for j in range(1, t.shape[1]):
        m = re.search(r"PISA (\d{4})$", str(ciclo[j]).strip())  # descarta las columnas de "Change ..."
        if not m or str(tipo[j]).strip() not in {"Score", "%", "S.E."}:
            continue
        valores = pd.to_numeric(t.iloc[fila_ciclo + 3:, j], errors="coerce")
        for pais, valor in zip(paises, valores):
            filas.append({
                "Location": pais, "Time": int(m.group(1)),
                "medida": str(medida[j]).split("\n")[0].strip(),
                "tipo": "se" if str(tipo[j]).strip() == "S.E." else "valor",
                "v": valor,
            })
    largo = pd.DataFrame(filas)
    return largo.pivot_table(index=["Location", "Time", "medida"], columns="tipo", values="v").reset_index()


HOJAS = {
    "Matematicas": ("Table I.B1.2a.41", "Table I.B1.2a.35"),
    "Lectura": ("Table I.B1.2a.40", "Table I.B1.2a.34"),
    "Ciencias": ("Table I.B1.2a.39", "Table I.B1.2a.33"),
}
NOMBRE_A_ISO = {v: k for k, v in PANEL.items()}

partes = []
for materia, (hoja_dist, hoja_bajo) in HOJAS.items():
    dist = leer_tendencia(hoja_dist)
    bajo = leer_tendencia(hoja_bajo)
    sel = lambda d, med: d.loc[d["medida"].eq(med) & d["Location"].isin(NOMBRE_A_ISO)].set_index(["Location", "Time"])[["valor", "se"]]
    percentiles = {"P10": "10th percentile", "P25": "25th percentile", "P50": "Median (50th percentile)",
                   "P75": "75th percentile", "P90": "90th percentile"}
    columnas = {}
    for corto, largo in percentiles.items():
        d = sel(dist, largo)
        columnas[corto], columnas[f"{corto}_se"] = d["valor"], d["se"]
    b2 = sel(bajo, "Below Level 2")
    columnas["pct_bajo_nivel2"], columnas["pct_bajo_nivel2_se"] = b2["valor"], b2["se"]
    tabla = pd.concat(columnas, axis=1).reset_index()
    tabla["Materia"] = materia
    partes.append(tabla)

pisa = pd.concat(partes, ignore_index=True)
pisa["ISO3_code"] = pisa["Location"].map(NOMBRE_A_ISO)
pisa = pisa.loc[pisa["Time"].isin(CICLOS)]

# Rejilla completa país × ciclo × materia: así los ciclos sin dato quedan visibles como filas vacías
rejilla = pd.MultiIndex.from_product([list(PANEL), CICLOS, list(HOJAS)], names=["ISO3_code", "Time", "Materia"])
pisa = pisa.set_index(["ISO3_code", "Time", "Materia"]).reindex(rejilla).reset_index()
pisa["Location"] = pisa["ISO3_code"].map(PANEL)
print("Filas (país × ciclo × materia):", len(pisa))
print("Países encontrados:", sorted(pisa["ISO3_code"].unique()))
assert pisa["ISO3_code"].nunique() == len(PANEL)

Filas (país × ciclo × materia): 168
Países encontrados: ['ARG', 'BRA', 'CHL', 'COL', 'CRI', 'MEX', 'PER', 'URY']


### Brecha P90 − P10

La brecha es la distancia, en puntos PISA, entre el estudiante del percentil 90 y el del percentil 10 de su país. Su error estándar se aproxima como $\sqrt{SE_{90}^2 + SE_{10}^2}$, una aproximación conservadora porque ignora la covarianza entre los dos percentiles.

Dentro de un mismo ciclo, el error de enlace entre ciclos se cancela (afecta igual a los dos percentiles), así que la brecha se puede comparar entre años sin sumarlo. En cambio, el **% bajo el nivel 2** sí depende del enlace entre escalas: sus cambios entre ciclos son algo más inciertos de lo que indica su error estándar.

In [5]:
pisa["brecha_P90_P10"] = pisa["P90"] - pisa["P10"]
pisa["brecha_P90_P10_se"] = np.sqrt(pisa["P90_se"] ** 2 + pisa["P10_se"] ** 2)

# ¿Qué ciclos faltan en el panel principal (2009-2025)?
faltan = (
    pisa.loc[pisa["Time"].ge(INICIO_PANEL) & pisa["brecha_P90_P10"].isna(), ["ISO3_code", "Time", "Materia"]]
    .sort_values(["ISO3_code", "Time"])
)
print("Celdas sin dato en 2009-2025 antes de exclusiones:")
print(faltan.to_string(index=False) if len(faltan) else "  ninguna")

Celdas sin dato en 2009-2025 antes de exclusiones:
ISO3_code  Time     Materia
      ARG  2015 Matematicas
      ARG  2015     Lectura
      ARG  2015    Ciencias


### Exclusión de Argentina 2015

La OCDE dejó los resultados nacionales de Argentina 2015 fuera de su informe principal y los considera no comparables: una reestructuración de la secundaria redujo la parte de los jóvenes de 15 años que cubría la muestra (su índice de cobertura, sección 3, cae a 0,55 frente a 0,80 en 2012 y 0,81 en 2018). Se deja el ciclo vacío y se marca con `excluido_motivo`, en lugar de borrar la fila.

In [6]:
pisa["excluido_motivo"] = [EXCLUIR.get((i, int(t)), "") for i, t in zip(pisa["ISO3_code"], pisa["Time"])]
medidas = [c for c in pisa.columns if c.startswith(("P10", "P25", "P50", "P75", "P90", "brecha", "pct_bajo"))]
excl = pisa["excluido_motivo"] != ""
print("¿La tabla de la OCDE trae dato para Argentina 2015?")
print(pisa.loc[excl, ["Materia", "brecha_P90_P10", "pct_bajo_nivel2"]].to_string(index=False))
print("-> La OCDE ya lo publica como 'm' (sin dato): su tabla de tendencias coincide con la decisión del equipo.")
pisa.loc[excl, medidas] = np.nan
print("Filas marcadas como excluidas:", excl.sum())

¿La tabla de la OCDE trae dato para Argentina 2015?
    Materia  brecha_P90_P10  pct_bajo_nivel2
Matematicas             NaN              NaN
    Lectura             NaN              NaN
   Ciencias             NaN              NaN
-> La OCDE ya lo publica como 'm' (sin dato): su tabla de tendencias coincide con la decisión del equipo.
Filas marcadas como excluidas: 3


## 3. Cobertura: ¿a qué parte de los jóvenes de 15 años representa PISA?

El **índice de cobertura 3** de la OCDE (tabla I.A2.1) es la proporción de todos los jóvenes de 15 años del país que representa la muestra de PISA. Quienes no están escolarizados (o están en grados muy bajos) quedan fuera. Si el índice sube con el tiempo, entran a la prueba jóvenes que antes estaban por fuera, que suelen venir de hogares con menos recursos: eso puede **aumentar la desigualdad medida aunque el sistema educativo mejore**.

In [7]:
t = cargar_raw("PISA2025_anexoA2_cobertura.xlsx", sheet_name="Table I.A2.1", header=None)
# La fila de ciclos es la que tiene una celda exactamente igual a "PISA 2025" (el título también menciona PISA)
fila_ciclo = next(i for i in range(len(t)) if t.iloc[i].astype(str).str.strip().eq("PISA 2025").any())
ciclo = t.iloc[fila_ciclo].ffill()
medida = t.iloc[fila_ciclo + 1].ffill()

cob = []
for j in range(1, t.shape[1]):
    if str(medida[j]).startswith("Coverage Index 3"):
        anio = int(re.search(r"(\d{4})", str(ciclo[j])).group(1))
        valores = pd.to_numeric(t.iloc[fila_ciclo + 2:, j], errors="coerce")
        nombres = t.iloc[fila_ciclo + 2:, 0].map(limpiar_nombre)
        cob.append(pd.DataFrame({"Location": nombres.values, "Time": anio, "cobertura_ci3": valores.values}))
cob = pd.concat(cob)
cob = cob.loc[cob["Location"].isin(NOMBRE_A_ISO)].assign(ISO3_code=lambda d: d["Location"].map(NOMBRE_A_ISO))
cob = cob.dropna(subset=["cobertura_ci3"]).drop_duplicates(["ISO3_code", "Time"])
print("Ciclos con cobertura:", sorted(cob["Time"].unique()))
cob.pivot_table(index="ISO3_code", columns="Time", values="cobertura_ci3").round(2)

Ciclos con cobertura: [np.int64(2003), np.int64(2006), np.int64(2009), np.int64(2012), np.int64(2015), np.int64(2018), np.int64(2022), np.int64(2025)]


Time,2003,2006,2009,2012,2015,2018,2022,2025
ISO3_code,,,,,,,,
ARG,NaN,0.79,0.69,0.80,0.55,0.81,0.84,0.87
BRA,0.55,0.55,0.61,0.70,0.72,0.65,0.76,0.76
CHL,NaN,0.79,0.85,0.85,0.79,0.87,0.86,0.95
COL,NaN,0.60,0.58,0.63,0.75,0.62,0.73,0.75
CRI,NaN,NaN,0.53,0.50,0.63,0.63,0.78,0.73
MEX,0.49,0.54,0.61,0.60,0.63,0.66,0.64,0.69
PER,NaN,NaN,0.73,0.72,0.74,0.73,0.86,0.86
URY,0.63,0.69,0.63,0.72,0.73,0.78,0.85,0.76


## 4. Unión con la demografía

Cada fila del panel lleva dos momentos demográficos:

- **El año del ciclo** (`TFR`, `IMR`...): el contexto del país cuando se aplicó la prueba.
- **El año de nacimiento de los evaluados** (`*_al_nacer`, ciclo − 15): los estudiantes de PISA tienen unos 15 años, así que la fecundidad y la mortalidad infantil de 15 años antes describen el contexto en que nació esa misma generación.

2025 es proyección en WPP2024 y queda marcado con `es_proyeccion`.

In [8]:
demo = al.loc[al["ISO3_code"].isin(PANEL), ["ISO3_code", "Time", "TFR", "IMR", "pct_births_adolescentes", "MedianAgePop", "es_proyeccion"]]
al_nacer = demo[["ISO3_code", "Time", "TFR", "IMR", "pct_births_adolescentes"]].copy()
al_nacer["Time"] = al_nacer["Time"] + 15
al_nacer = al_nacer.rename(columns={c: f"{c}_al_nacer" for c in ["TFR", "IMR", "pct_births_adolescentes"]})

panel = (
    pisa.merge(cob[["ISO3_code", "Time", "cobertura_ci3"]], on=["ISO3_code", "Time"], how="left", validate="many_to_one")
    .merge(demo, on=["ISO3_code", "Time"], how="left", validate="many_to_one")
    .merge(al_nacer, on=["ISO3_code", "Time"], how="left", validate="many_to_one")
)
panel["Pais"] = panel["ISO3_code"].map(PANEL)
panel["en_panel_principal"] = panel["Time"].ge(INICIO_PANEL)

orden = ["ISO3_code", "Pais", "Time", "Materia", "en_panel_principal", "es_proyeccion", "excluido_motivo",
         "brecha_P90_P10", "brecha_P90_P10_se", "pct_bajo_nivel2", "pct_bajo_nivel2_se",
         "P10", "P10_se", "P25", "P25_se", "P50", "P50_se", "P75", "P75_se", "P90", "P90_se", "cobertura_ci3",
         "TFR", "IMR", "pct_births_adolescentes", "MedianAgePop",
         "TFR_al_nacer", "IMR_al_nacer", "pct_births_adolescentes_al_nacer"]
panel = panel[orden].sort_values(["Materia", "ISO3_code", "Time"]).reset_index(drop=True)
print("Panel:", panel.shape)
panel.loc[panel["Materia"].eq("Matematicas") & panel["ISO3_code"].eq("COL")].round(2)

Panel: (168, 29)


,ISO3_code,Pais,Time,Materia,en_panel_principal,es_proyeccion,excluido_motivo,brecha_P90_P10,brecha_P90_P10_se,pct_bajo_nivel2,pct_bajo_nivel2_se,P10,P10_se,P25,P25_se,P50,P50_se,P75,P75_se,P90,P90_se,cobertura_ci3,TFR,IMR,pct_births_adolescentes,MedianAgePop,TFR_al_nacer,IMR_al_nacer,pct_births_adolescentes_al_nacer
133,COL,Colombia,2006,Matematicas,False,False,,224.22,6.80,71.90,1.61,257.54,5.64,311.32,4.92,369.16,4.01,428.27,4.59,481.77,3.80,0.60,2.24,17.09,21.70,24.96,2.98,27.83,17.91
134,COL,Colombia,2009,Matematicas,True,False,,192.58,6.66,70.45,1.59,286.27,5.15,329.72,4.04,379.29,3.36,430.81,3.43,478.85,4.23,0.58,2.10,15.55,22.48,26.07,2.94,25.13,17.92
135,COL,Colombia,2012,Matematicas,True,False,,188.85,6.26,73.82,1.43,285.19,4.02,325.79,2.77,372.60,2.90,423.19,3.63,474.04,4.80,0.63,1.90,14.14,23.03,27.29,2.74,22.64,18.97
136,COL,Colombia,2015,Matematicas,True,False,,199.03,4.55,66.29,1.18,292.99,3.12,335.47,2.90,386.39,2.61,441.38,2.69,492.01,3.31,0.75,1.77,12.83,20.89,28.57,2.57,20.72,20.78
137,COL,Colombia,2018,Matematicas,True,False,,208.73,5.92,65.39,1.57,290.08,3.86,334.74,3.53,387.42,3.53,444.52,3.80,498.81,4.49,0.62,1.71,11.64,19.31,29.73,2.45,18.70,20.98
138,COL,Colombia,2022,Matematicas,True,False,,187.35,5.35,71.22,1.55,293.26,3.09,331.66,3.19,377.54,3.52,429.32,3.75,480.60,4.37,0.73,1.66,10.19,16.96,31.18,2.20,16.59,22.13
139,COL,Colombia,2025,Matematicas,True,True,,194.07,5.43,70.96,1.56,286.86,2.46,326.97,2.51,376.13,3.13,430.01,3.69,480.93,4.84,0.75,1.62,9.28,15.83,32.50,2.01,15.14,22.64


## 5. Comprobaciones y guardado

In [9]:
mat = panel.loc[panel["Materia"].eq("Matematicas") & panel["en_panel_principal"]]
# Los 8 países tienen dato en todos los ciclos 2009-2025, salvo la exclusión decidida
sin_dato = mat.loc[mat["brecha_P90_P10"].isna(), ["ISO3_code", "Time"]]
assert set(map(tuple, sin_dato.values)) <= set(EXCLUIR), sin_dato
assert mat[["TFR", "IMR", "TFR_al_nacer", "IMR_al_nacer"]].notna().all().all()
assert not panel.duplicated(["ISO3_code", "Time", "Materia"]).any()
con_dato = panel.dropna(subset=["P10", "P90"])
assert (con_dato["P90"] > con_dato["P10"]).all()
print("Celdas de cobertura vacías en el panel principal (matemáticas):", mat["cobertura_ci3"].isna().sum())

_ = guardar_processed(al, "al_wpp_1950_2100.csv")
_ = guardar_processed(panel, "al_panel_pisa.csv")

Celdas de cobertura vacías en el panel principal (matemáticas): 0
Guardado: data\processed\al_wpp_1950_2100.csv (7,701 filas)
Guardado: data\processed\al_panel_pisa.csv (168 filas)
